In [4]:
# Day 3 - Fundamental Scoring System
import pandas as pd
import numpy as np
import requests
import pickle
import time
import warnings
from bs4 import BeautifulSoup
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [1]:
# ── CELL 2: SCRAPER FUNCTIONS (FINAL VERSION) ───────────────
# Key fix: scraper checks if P&L table has real data columns
# before accepting consolidated page. Falls back to standalone
# automatically. This fixed WONDERLA, GULFPETRO, CREDITACC, AAVAS.
 
def extract_table(table):
    """Extract any table from Screener.in cleanly"""
    try:
        thead   = table.find('thead')
        columns = []
        if thead:
            for th in thead.find_all('th'):
                text = th.get_text(strip=True)
                columns.append(text if text else 'Metric')
 
        tbody = table.find('tbody')
        if not tbody:
            return None
 
        rows = tbody.find_all('tr')
        data = {}
 
        for row in rows:
            cells       = row.find_all('td')
            if not cells:
                continue
            metric_name = cells[0].get_text(strip=True)
            metric_name = metric_name.replace('+', '').strip()
 
            skip_keywords = ['Raw PDF', 'PDF', 'Source']
            if any(kw in metric_name for kw in skip_keywords):
                continue
 
            values = []
            for cell in cells[1:]:
                val = cell.get_text(strip=True)
                val = val.replace(',', '').replace('%', '').strip()
                try:
                    values.append(float(val))
                except:
                    values.append(val if val else None)
 
            data[metric_name] = values
 
        if not data:
            return None
 
        col_names = columns[1:] if len(columns) > 1 else list(range(len(values)))
        df        = pd.DataFrame(data, index=col_names).T
        return df
 
    except Exception as e:
        print(f"Table extraction error: {e}")
        return None
 
 
def scrape_screener_complete(symbol):
    """
    Scrape all sections from Screener.in
    Tries consolidated first — checks if P&L has real data columns
    Falls back to standalone if consolidated has empty headers
    """
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
 
        urls = [
            f"https://www.screener.in/company/{symbol}/consolidated/",
            f"https://www.screener.in/company/{symbol}/",
        ]
 
        soup = None
        for url in urls:
            response = requests.get(url, headers=headers, timeout=15)
            if response.status_code != 200:
                continue
 
            temp_soup = BeautifulSoup(response.text, 'html.parser')
 
            # Check if P&L table has actual data columns (not empty)
            has_data = False
            for section in temp_soup.find_all('section'):
                heading = section.find('h2')
                if heading and 'Profit' in heading.text:
                    table = section.find('table')
                    if table:
                        thead = table.find('thead')
                        if thead:
                            ths = thead.find_all('th')
                            # More than 1 th = real data exists
                            if len(ths) > 1:
                                has_data = True
                    break
 
            if has_data:
                soup = temp_soup
                break
 
        if soup is None:
            return None
 
        target_sections = [
            'Quarterly Results',
            'Profit & Loss',
            'Balance Sheet',
            'Cash Flows',
            'Ratios',
            'Shareholding Pattern',
        ]
 
        result = {}
 
        for section in soup.find_all('section'):
            heading = section.find('h2')
            if not heading:
                continue
            section_name = heading.text.strip()
            if section_name not in target_sections:
                continue
            table = section.find('table')
            if not table:
                continue
            df = extract_table(table)
            if df is not None:
                result[section_name] = df
 
        # Extract sector info from Peer section
        for section in soup.find_all('section'):
            heading = section.find('h2')
            if heading and 'Peer' in heading.text:
                links       = section.find_all('a')
                sector_info = {}
                titles      = ['Broad Sector', 'Sector',
                               'Broad Industry', 'Industry']
                for link in links:
                    title = link.get('title', '')
                    if title in titles:
                        sector_info[title] = link.text.strip()
                result['Sector Info'] = sector_info
                break
 
        return result if result else None
 
    except Exception as e:
        print(f"Error scraping {symbol}: {e}")
        return None
 
 
print("Scraper functions loaded!")

Scraper functions loaded!


In [2]:
# ── CELL 3: STOCK LIST ──────────────────────────────────────
stocks_to_fetch = [
    # Large Cap IT
    'TCS', 'INFY', 'HDFCBANK', 'RELIANCE', 'WIPRO',
    # Mid Cap IT
    'PERSISTENT', 'MPHASIS', 'LTTS', 'COFORGE', 'HAPPSTMNDS',
    # Finance
    'CHOLAFIN', 'MUTHOOTFIN', 'MANAPPURAM', 'IIFL', 'CREDITACC',
    # Manufacturing
    'GARFIBRES', 'SUPRAJIT', 'HIMATSEIDE', 'IGPL',
    # Defence
    'HAL', 'BEL', 'BHEL', 'MIDHANI', 'PARAS',
    # Chemicals
    'VINATIORGA', 'CLEAN', 'FINEORG', 'GALAXYSURF',
    # Finance/Housing
    'AAVAS',
    # Pharma
    'SUNPHARMA', 'DIVISLAB', 'APLLTD', 'GRANULES', 'IPCALAB',
    # Consumer
    'TITAN', 'HAVELLS', 'VGUARD', 'SYMPHONY', 'WONDERLA',
    # Agrochem
    'PIIND', 'RALLIS', 'DHANUKA',
    # Others
    'DYNPRO', 'GULFPETRO', 'SUVIDHAA', 'ABSLAMC',
]
print(f"Total stocks to fetch: {len(stocks_to_fetch)}")

Total stocks to fetch: 46


In [5]:
# ── CELL 4: FETCH ALL STOCKS ────────────────────────────────
all_stock_data = {}
failed_stocks  = []
 
print(f"Starting fetch for {len(stocks_to_fetch)} stocks...")
print("This will take 10-15 minutes. Please wait...\n")
 
for i, symbol in enumerate(stocks_to_fetch):
    try:
        print(f"[{i+1}/{len(stocks_to_fetch)}] Fetching {symbol}...", end=" ")
        data = scrape_screener_complete(symbol)
        if data:
            all_stock_data[symbol] = data
            sections = [k for k in data.keys() if k != 'Sector Info']
            print(f"✓ ({len(sections)} sections)")
        else:
            failed_stocks.append(symbol)
            print(f"✗ Failed")
        time.sleep(2)
    except Exception as e:
        failed_stocks.append(symbol)
        print(f"✗ Error: {e}")
        time.sleep(2)
 
print(f"\nSuccessfully fetched : {len(all_stock_data)} stocks")
print(f"Failed               : {len(failed_stocks)} stocks")
if failed_stocks:
    print(f"Failed stocks        : {failed_stocks}")
 
# Save raw data
with open('../data/raw_stock_data.pkl', 'wb') as f:
    pickle.dump(all_stock_data, f)
print("Saved raw_stock_data.pkl ✅")
 
 
# ── CELL 5: LOAD RAW DATA (use instead of Cell 4 if already fetched)
with open('../data/raw_stock_data.pkl', 'rb') as f:
    all_stock_data = pickle.load(f)
print(f"Loaded: {len(all_stock_data)} stocks")

Starting fetch for 46 stocks...
This will take 10-15 minutes. Please wait...

[1/46] Fetching TCS... ✓ (6 sections)
[2/46] Fetching INFY... ✓ (6 sections)
[3/46] Fetching HDFCBANK... ✓ (6 sections)
[4/46] Fetching RELIANCE... ✓ (6 sections)
[5/46] Fetching WIPRO... ✓ (6 sections)
[6/46] Fetching PERSISTENT... ✓ (6 sections)
[7/46] Fetching MPHASIS... ✓ (6 sections)
[8/46] Fetching LTTS... ✓ (6 sections)
[9/46] Fetching COFORGE... ✓ (6 sections)
[10/46] Fetching HAPPSTMNDS... ✓ (6 sections)
[11/46] Fetching CHOLAFIN... ✓ (6 sections)
[12/46] Fetching MUTHOOTFIN... ✓ (6 sections)
[13/46] Fetching MANAPPURAM... ✓ (6 sections)
[14/46] Fetching IIFL... ✓ (6 sections)
[15/46] Fetching CREDITACC... ✓ (6 sections)
[16/46] Fetching GARFIBRES... ✓ (6 sections)
[17/46] Fetching SUPRAJIT... ✓ (6 sections)
[18/46] Fetching HIMATSEIDE... ✓ (6 sections)
[19/46] Fetching IGPL... ✓ (6 sections)
[20/46] Fetching HAL... ✓ (6 sections)
[21/46] Fetching BEL... ✓ (6 sections)
[22/46] Fetching BHEL... ✓ (6 s

In [6]:
# ── CELL 6: EXTRACT ALL METRICS (FINAL VERSION) ─────────────
# Key decisions made during debugging:
# 1. Financial companies use different P&L row names:
#    Revenue (not Sales), Borrowing (not Borrowings),
#    Financing Margin % (not OPM %)
# 2. ROE from Ratios section used for financial companies
#    (more accurate than calculated ROE from balance sheet)
# 3. Final ROE = ROE from Ratios if available, else calculated ROE
# 4. Revenue CAGR Max = best available CAGR (up to 10Y)
#    used when 5Y or 10Y data not available
 
def extract_all_metrics(symbol, data):
    """
    Extract all 50 fundamental metrics from raw Screener.in data
    Handles both regular and financial company row names
    """
    m = {'Symbol': symbol}
 
    try:
        # ── SECTOR INFO ──────────────────────────────────────
        sector_info       = data.get('Sector Info', {})
        m['Sector']       = sector_info.get('Sector',       None)
        m['Industry']     = sector_info.get('Industry',     None)
        m['Broad Sector'] = sector_info.get('Broad Sector', None)
 
        # Detect financial company by sector name
        financial_keywords = [
            'Financial', 'Banking', 'Insurance',
            'NBFC', 'Lending', 'Microfinance'
        ]
        is_financial = any(
            kw in str(m.get('Sector', ''))
            for kw in financial_keywords
        )
 
        # Row name mapping — financial companies differ
        if is_financial:
            revenue_row = 'Revenue'
            margin_row  = 'Financing Margin %'
            profit_row  = 'Financing Profit'
            debt_row    = 'Borrowing'
        else:
            revenue_row = 'Sales'
            margin_row  = 'OPM %'
            profit_row  = 'Operating Profit'
            debt_row    = 'Borrowings'
 
        # ── ANNUAL P&L → Long term growth ────────────────────
        pl = data.get('Profit & Loss')
        if pl is not None:
            pl_clean = pl.drop(columns=['TTM'], errors='ignore')
 
            # Revenue CAGR
            if revenue_row in pl_clean.index:
                sales = pl_clean.loc[revenue_row].dropna()
 
                if len(sales) >= 6:
                    m['Revenue CAGR 5Y']  = round(
                        ((sales.iloc[-1] / sales.iloc[-6]) ** (1/5) - 1) * 100, 2
                    )
                if len(sales) >= 11:
                    m['Revenue CAGR 10Y'] = round(
                        ((sales.iloc[-1] / sales.iloc[-11]) ** (1/10) - 1) * 100, 2
                    )
                # Max CAGR — uses best available years up to 10
                if len(sales) >= 3:
                    years_available         = min(len(sales) - 1, 10)
                    base                    = sales.iloc[-years_available - 1]
                    if base > 0:
                        m['Revenue CAGR Max']   = round(
                            ((sales.iloc[-1] / base) ** (1/years_available) - 1) * 100, 2
                        )
                        m['Revenue CAGR Years'] = years_available
 
            # OPM / Financing Margin
            if margin_row in pl_clean.index:
                opm              = pl_clean.loc[margin_row].dropna()
                m['Avg OPM 5Y']  = round(opm.iloc[-5:].mean(),  2) if len(opm) >= 5  else (round(opm.mean(), 2) if len(opm) > 0 else None)
                m['Avg OPM 10Y'] = round(opm.iloc[-10:].mean(), 2) if len(opm) >= 10 else None
 
            # Profit CAGR (Net Profit row same for all companies)
            if 'Net Profit' in pl_clean.index:
                profit = pl_clean.loc['Net Profit'].dropna()
 
                if len(profit) >= 6 and profit.iloc[-6] > 0:
                    m['Profit CAGR 5Y']  = round(
                        ((profit.iloc[-1] / profit.iloc[-6]) ** (1/5) - 1) * 100, 2
                    )
                if len(profit) >= 11 and profit.iloc[-11] > 0:
                    m['Profit CAGR 10Y'] = round(
                        ((profit.iloc[-1] / profit.iloc[-11]) ** (1/10) - 1) * 100, 2
                    )
                # Max CAGR
                if len(profit) >= 3:
                    years_available      = min(len(profit) - 1, 10)
                    base                 = profit.iloc[-years_available - 1]
                    if base > 0:
                        m['Profit CAGR Max']   = round(
                            ((profit.iloc[-1] / base) ** (1/years_available) - 1) * 100, 2
                        )
                        m['Profit CAGR Years'] = years_available
 
        # ── QUARTERLY DATA → Recent momentum ─────────────────
        qr = data.get('Quarterly Results')
        if qr is not None:
            qr_clean = qr.drop(columns=['TTM'], errors='ignore')
 
            # Revenue quarterly
            if revenue_row in qr_clean.index:
                sales_q = qr_clean.loc[revenue_row].dropna()
 
                if len(sales_q) >= 4:
                    m['TTM Revenue']             = round(sales_q.iloc[-4:].sum(), 2)
                if len(sales_q) >= 5:
                    latest                       = sales_q.iloc[-1]
                    year_ago                     = sales_q.iloc[-5]
                    if year_ago > 0:
                        m['Revenue YoY Q']       = round(
                            (latest - year_ago) / year_ago * 100, 2
                        )
                # Consecutive YoY improvements
                yoy_improvements = 0
                for i in range(1, min(5, len(sales_q) - 4)):
                    curr = sales_q.iloc[-i]
                    prev = sales_q.iloc[-i - 4]
                    if prev > 0 and curr > prev:
                        yoy_improvements += 1
                    else:
                        break
                m['Revenue Consecutive YoY'] = yoy_improvements
 
                # Revenue acceleration
                if len(sales_q) >= 9:
                    recent_yoy               = (sales_q.iloc[-1] - sales_q.iloc[-5]) / sales_q.iloc[-5] * 100
                    previous_yoy             = (sales_q.iloc[-5] - sales_q.iloc[-9]) / sales_q.iloc[-9] * 100
                    m['Revenue Accelerating'] = recent_yoy > previous_yoy
 
            # Profit quarterly
            if 'Net Profit' in qr_clean.index:
                profit_q = qr_clean.loc['Net Profit'].dropna()
 
                if len(profit_q) >= 4:
                    m['TTM Profit']          = round(profit_q.iloc[-4:].sum(), 2)
                if len(profit_q) >= 5:
                    latest                   = profit_q.iloc[-1]
                    year_ago                 = profit_q.iloc[-5]
                    if year_ago > 0:
                        m['Profit YoY Q']    = round(
                            (latest - year_ago) / year_ago * 100, 2
                        )
                m['Profit Positive Q']       = int((profit_q > 0).sum())
                m['Total Quarters']          = len(profit_q)
 
            # Margin quarterly
            if margin_row in qr_clean.index:
                opm_q = qr_clean.loc[margin_row].dropna()
 
                m['Latest OPM Q']            = opm_q.iloc[-1]  if len(opm_q) > 0 else None
                m['Avg OPM 4Q']              = round(opm_q.iloc[-4:].mean(), 2) if len(opm_q) >= 4 else None
                if len(opm_q) >= 5:
                    m['Margin Improving']    = bool(opm_q.iloc[-1] >= opm_q.iloc[-5])
 
            # EPS quarterly
            if 'EPS in Rs' in qr_clean.index:
                eps_q = qr_clean.loc['EPS in Rs'].dropna()
 
                m['Latest EPS Q']            = eps_q.iloc[-1] if len(eps_q) > 0 else None
                if len(eps_q) >= 5:
                    latest                   = eps_q.iloc[-1]
                    year_ago                 = eps_q.iloc[-5]
                    if year_ago > 0:
                        m['EPS YoY Growth']  = round(
                            (latest - year_ago) / year_ago * 100, 2
                        )
 
        # ── BALANCE SHEET ─────────────────────────────────────
        bs = data.get('Balance Sheet')
        if bs is not None:
            if debt_row in bs.index:
                debt                         = bs.loc[debt_row].dropna()
                m['Latest Debt']             = debt.iloc[-1] if len(debt) > 0 else None
                if len(debt) >= 4:
                    m['Debt Reducing']       = bool(debt.iloc[-1] < debt.iloc[-4])
 
            if 'Reserves' in bs.index and 'Equity Capital' in bs.index:
                reserves                     = bs.loc['Reserves'].dropna()
                eq_cap                       = bs.loc['Equity Capital'].dropna()
                if len(reserves) > 0 and len(eq_cap) > 0:
                    equity                   = reserves.iloc[-1] + eq_cap.iloc[-1]
                    m['Latest Equity']       = round(equity, 2)
                    if m.get('TTM Profit') and equity > 0:
                        m['ROE']             = round(m['TTM Profit'] / equity * 100, 2)
                    if m.get('Latest Debt') is not None and equity > 0:
                        m['Debt to Equity']  = round(m['Latest Debt'] / equity, 2)
 
        # ── CASH FLOWS ───────────────────────────────────────
        cf = data.get('Cash Flows')
        if cf is not None:
            if 'Cash from Operating Activity' in cf.index:
                op_cf                        = cf.loc['Cash from Operating Activity'].dropna()
                m['Latest Operating CF']     = op_cf.iloc[-1] if len(op_cf) > 0 else None
                m['CF Positive Years']       = int((op_cf > 0).sum())
                m['CF Total Years']          = len(op_cf)
                if len(op_cf) >= 4:
                    m['CF Growing']          = bool(op_cf.iloc[-1] > op_cf.iloc[-4])
 
        # ── RATIOS ───────────────────────────────────────────
        ratios = data.get('Ratios')
        if ratios is not None:
            if 'ROCE %' in ratios.index:
                roce                         = ratios.loc['ROCE %'].dropna()
                m['Latest ROCE']             = roce.iloc[-1] if len(roce) > 0 else None
                if len(roce) >= 4:
                    m['ROCE Improving']      = bool(roce.iloc[-1] > roce.iloc[-4])
                if len(roce) >= 5:
                    m['Avg ROCE 5Y']         = round(roce.iloc[-5:].mean(), 2)
 
            # ROE from Ratios — only available for financial companies
            # on Screener.in, shown as 'ROE %'
            if 'ROE %' in ratios.index:
                roe_series                   = ratios.loc['ROE %'].dropna()
                m['ROE from Ratios']         = roe_series.iloc[-1] if len(roe_series) > 0 else None
                m['ROE Avg 5Y']              = round(roe_series.iloc[-5:].mean(), 2) if len(roe_series) >= 5 else None
                m['ROE Improving']           = bool(roe_series.iloc[-1] > roe_series.iloc[-4]) if len(roe_series) >= 4 else None
 
        # ── SHAREHOLDING ─────────────────────────────────────
        sh = data.get('Shareholding Pattern')
        if sh is not None:
            if 'Promoters' in sh.index:
                promoter                     = sh.loc['Promoters'].dropna()
                m['Promoter Holding']        = promoter.iloc[-1] if len(promoter) > 0 else None
                if len(promoter) >= 5:
                    m['Promoter Change 4Q']  = round(promoter.iloc[-1] - promoter.iloc[-5], 2)
                if len(promoter) >= 9:
                    m['Promoter Change 8Q']  = round(promoter.iloc[-1] - promoter.iloc[-9], 2)
 
            if 'FIIs' in sh.index:
                fii                          = sh.loc['FIIs'].dropna()
                m['FII Holding']             = fii.iloc[-1] if len(fii) > 0 else None
                if len(fii) >= 5:
                    m['FII Change 4Q']       = round(fii.iloc[-1] - fii.iloc[-5], 2)
 
            if 'DIIs' in sh.index:
                dii                          = sh.loc['DIIs'].dropna()
                m['DII Holding']             = dii.iloc[-1] if len(dii) > 0 else None
                if len(dii) >= 5:
                    m['DII Change 4Q']       = round(dii.iloc[-1] - dii.iloc[-5], 2)
 
    except Exception as e:
        print(f"Error extracting {symbol}: {e}")
 
    return m
 
 
# Test on TCS
print("Testing on TCS:")
tcs_metrics = extract_all_metrics('TCS', all_stock_data['TCS'])
for key, value in tcs_metrics.items():
    print(f"  {key:30} : {value}")

Testing on TCS:
  Symbol                         : TCS
  Sector                         : Information Technology
  Industry                       : Computers - Software & Consulting
  Broad Sector                   : Information Technology
  Revenue CAGR 5Y                : 10.22
  Revenue CAGR 10Y               : 10.43
  Revenue CAGR Max               : 10.43
  Revenue CAGR Years             : 10
  Avg OPM 5Y                     : 27.0
  Avg OPM 10Y                    : 27.0
  Profit CAGR 5Y                 : 8.5
  Profit CAGR 10Y                : 9.3
  Profit CAGR Max                : 9.3
  Profit CAGR Years              : 10
  TTM Revenue                    : 260802.0
  Revenue YoY Q                  : 4.87
  Revenue Consecutive YoY        : 4
  Revenue Accelerating           : False
  TTM Profit                     : 47963.0
  Profit YoY Q                   : -13.85
  Profit Positive Q              : 13
  Total Quarters                 : 13
  Latest OPM Q                   : 27.0
 

In [7]:
# ── CELL 7: EXTRACT FOR ALL STOCKS + ADD FINAL ROE ──────────
all_metrics = []
 
for symbol, data in all_stock_data.items():
    metrics = extract_all_metrics(symbol, data)
    all_metrics.append(metrics)
    print(f"✓ {symbol}")
 
metrics_df = pd.DataFrame(all_metrics)
 
# Final ROE — unified column
# Uses ROE from Ratios (financial co) if available
# Falls back to calculated ROE from balance sheet
def get_final_roe(row):
    if pd.notna(row.get('ROE from Ratios')):
        return row['ROE from Ratios']
    elif pd.notna(row.get('ROE')):
        return row['ROE']
    return None
 
metrics_df['Final ROE'] = metrics_df.apply(get_final_roe, axis=1)
 
print(f"\nTotal stocks     : {len(metrics_df)}")
print(f"Metrics per stock: {len(metrics_df.columns)}")

✓ TCS
✓ INFY
✓ HDFCBANK
✓ RELIANCE
✓ WIPRO
✓ PERSISTENT
✓ MPHASIS
✓ LTTS
✓ COFORGE
✓ HAPPSTMNDS
✓ CHOLAFIN
✓ MUTHOOTFIN
✓ MANAPPURAM
✓ IIFL
✓ CREDITACC
✓ GARFIBRES
✓ SUPRAJIT
✓ HIMATSEIDE
✓ IGPL
✓ HAL
✓ BEL
✓ BHEL
✓ MIDHANI
✓ PARAS
✓ VINATIORGA
✓ CLEAN
✓ FINEORG
✓ GALAXYSURF
✓ SUNPHARMA
✓ DIVISLAB
✓ APLLTD
✓ GRANULES
✓ IPCALAB
✓ TITAN
✓ HAVELLS
✓ VGUARD
✓ SYMPHONY
✓ WONDERLA
✓ PIIND
✓ RALLIS
✓ DHANUKA
✓ DYNPRO
✓ GULFPETRO
✓ SUVIDHAA
✓ ABSLAMC

Total stocks     : 45
Metrics per stock: 50


In [21]:
# ── CELL 8: SURVIVAL FILTER ─────────────────────────────────
# 6 rules that eliminate dangerous stocks before scoring
# Financial companies skipped from Debt/Equity rule
# Context-aware: uses Final ROE, CF consistency, promoter data
 
def apply_survival_filter(df):
    results = []

    for _, row in df.iterrows():
        symbol   = row['Symbol']
        reasons  = []
        passed   = True
        sector   = str(row.get('Sector', ''))

        # ── Company type detection ──────────────────────────
        financial_sectors = [
            'Financial Services', 'Banking', 'Insurance'
        ]
        is_financial = any(s in sector for s in financial_sectors)

        promoter = float(row.get('Promoter Holding') or 0)
        fii      = float(row.get('FII Holding')      or 0)
        dii      = float(row.get('DII Holding')      or 0)

        # MNC = foreign parent owns company
        # Either via high FII or high DII (routed differently)
        is_mnc = (promoter < 5 and (fii > 40 or dii > 40))

        # Institutional grade = low promoter but
        # very high combined institutional ownership
        # Means institutions have done their due diligence
        is_institutional = (
            promoter < 20 and
            (fii + dii) > 65
        )

        # Lumpy CF sectors
        lumpy_cf_sectors = [
            'Capital Goods',
            'Consumer Durables',
            'Consumer Services',
        ]
        is_lumpy_cf = any(s in sector for s in lumpy_cf_sectors)

        # ── Rule 1: Promoter Holding < 30% ──────────────────
        # Skip for MNC and institutional grade companies
        if not is_mnc and not is_institutional:
            if pd.notna(row.get('Promoter Holding')):
                if row['Promoter Holding'] < 30:
                    passed = False
                    reasons.append(
                        f"Low promoter: {row['Promoter Holding']}%"
                    )

        # ── Rule 2: Debt/Equity > 2 ─────────────────────────
        if not is_financial:
            if pd.notna(row.get('Debt to Equity')):
                if row['Debt to Equity'] > 2:
                    passed = False
                    reasons.append(
                        f"High D/E: {row['Debt to Equity']:.2f}"
                    )

        # ── Rule 3: Negative TTM Profit ──────────────────────
        if pd.notna(row.get('TTM Profit')):
            if row['TTM Profit'] <= 0:
                passed = False
                reasons.append(f"Negative TTM profit")

        # ── Rule 4: Cash Flow Consistency ───────────────────
        # Skip for financial companies
        # Relax to 60% for lumpy CF sectors
        # For Consumer Durables — use profit consistency
        # instead of CF (jewellery/retail = inventory heavy)
        if not is_financial:
            if (pd.notna(row.get('CF Positive Years')) and
                pd.notna(row.get('CF Total Years'))    and
                row['CF Total Years'] > 0):
        
                cf_ratio = row['CF Positive Years'] / row['CF Total Years']
        
                # Consumer Durables — check profit consistency
                # instead of CF (inventory intensive business)
                if 'Consumer Durables' in sector:
                    # Use profit positive quarters as proxy
                    if (pd.notna(row.get('Profit Positive Q')) and
                        pd.notna(row.get('Total Quarters'))    and
                        row['Total Quarters'] > 0):
                        profit_ratio = row['Profit Positive Q'] / row['Total Quarters']
                        if profit_ratio < 0.8:
                            passed = False
                            reasons.append(
                                f"Inconsistent profits: {row['Profit Positive Q']}"
                                f"/{row['Total Quarters']} quarters"
                            )
                    # Skip CF check for Consumer Durables

                else:
                    cf_threshold = 0.60 if is_lumpy_cf else 0.70
                    if cf_ratio < cf_threshold:
                        passed = False
                        reasons.append(
                            f"Poor CF: {row['CF Positive Years']}"
                            f"/{row['CF Total Years']} years"
                            f" (need {int(cf_threshold*100)}%)"
                        )

        # ── Rule 5: Promoter Selling > 5% in 4Q ─────────────
        # Skip for MNC and institutional grade companies
        if not is_mnc and not is_institutional:
            if pd.notna(row.get('Promoter Change 4Q')):
                if row['Promoter Change 4Q'] < -5:
                    passed = False
                    reasons.append(
                        f"Promoter selling: {row['Promoter Change 4Q']}% in 4Q"
                    )

        # ── Rule 6: Profit Positive Quarters < 80% ──────────
        if (pd.notna(row.get('Profit Positive Q')) and
            pd.notna(row.get('Total Quarters'))    and
            row['Total Quarters'] > 0):
            profit_ratio = row['Profit Positive Q'] / row['Total Quarters']
            if profit_ratio < 0.8:
                passed = False
                reasons.append(
                    f"Inconsistent profits: {row['Profit Positive Q']}"
                    f"/{row['Total Quarters']} quarters"
                )

        # Tag for reporting
        tags = []
        if is_mnc:          tags.append('MNC')
        if is_institutional: tags.append('INST')
        if is_financial:    tags.append('FIN')
        if is_lumpy_cf:     tags.append('LUMPY_CF')

        results.append({
            'Symbol'          : symbol,
            'Sector'          : row.get('Sector'),
            'Is Financial'    : is_financial,
            'Is MNC'          : is_mnc,
            'Is Institutional': is_institutional,
            'Is Lumpy CF'     : is_lumpy_cf,
            'Tags'            : ', '.join(tags),
            'Passed Survival' : passed,
            'Reason'          : ' | '.join(reasons) if reasons
                                else 'All checks passed'
        })

    return pd.DataFrame(results)


# Re-run
survival_results = apply_survival_filter(metrics_df)
passed           = survival_results[survival_results['Passed Survival'] == True]
failed           = survival_results[survival_results['Passed Survival'] == False]

print(f"Total stocks  : {len(survival_results)}")
print(f"Passed filter : {len(passed)}")
print(f"Failed filter : {len(failed)}")

print(f"\n=== FAILED STOCKS ===")
for _, row in failed.iterrows():
    print(f"  {row['Symbol']:15} → {row['Reason']}")

print(f"\n=== PASSED STOCKS ===")
for _, row in passed.sort_values('Symbol').iterrows():
    tag = f" [{row['Tags']}]" if row['Tags'] else ""
    print(f"  {row['Symbol']:15} {row['Sector']}{tag}")

Total stocks  : 45
Passed filter : 37
Failed filter : 8

=== FAILED STOCKS ===
  MPHASIS         → Promoter selling: -9.64% in 4Q
  IIFL            → Low promoter: 24.85%
  IGPL            → Negative TTM profit | Inconsistent profits: 9/13 quarters
  BHEL            → Inconsistent profits: 9/13 quarters
  CLEAN           → Promoter selling: -24.01% in 4Q
  DYNPRO          → Low promoter: 29.42%
  GULFPETRO       → Poor CF: 6/12 years (need 70%) | Promoter selling: -6.87% in 4Q
  SUVIDHAA        → Negative TTM profit | Inconsistent profits: 1/13 quarters

=== PASSED STOCKS ===
  ABSLAMC         Financial Services [FIN]
  APLLTD          Healthcare
  BEL             Capital Goods [LUMPY_CF]
  CHOLAFIN        Financial Services [FIN]
  COFORGE         Information Technology [MNC, INST]
  CREDITACC       Financial Services [FIN]
  DHANUKA         Chemicals
  DIVISLAB        Healthcare
  FINEORG         Chemicals
  GALAXYSURF      Chemicals
  GARFIBRES       Textiles
  GRANULES        Healt

In [22]:
#cell 9

# Save final results
survival_results.to_csv('../data/survival_filter_results.csv', index=False)
metrics_df.to_csv('../data/fundamental_metrics.csv', index=False)

with open('../data/raw_stock_data.pkl', 'wb') as f:
    pickle.dump(all_stock_data, f)

print(f"Total stocks     : {len(metrics_df)}")
print(f"Metrics per stock: {len(metrics_df.columns)}")
print(f"Passed survival  : {len(passed)}")
print(f"Failed survival  : {len(failed)}")
print(f"All files saved  ✅")

Total stocks     : 45
Metrics per stock: 50
Passed survival  : 37
Failed survival  : 8
All files saved  ✅
